# Advanced LLM Control with Groq
Day 1 — Advanced notebook with configurable model, system prompt, temperature, JSON mode, and optional log saving.

## Imports

In [28]:
import os
import json
import textwrap
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

True

## Configuration
Edit the settings below to control model behaviour.

In [ ]:
# ── Model settings ────────────────────────────────────────────
MODEL       = "llama-3.3-70b-versatile"   # Groq model to use
TEMPERATURE = 0.8                       # Lower = more deterministic
MAX_TOKENS  = 300                       # Maximum completion tokens
JSON_MODE   = False                     # Set True to force JSON output
SAVE_LOG    = True                     # Set True to save request/response to logs/

# ── System prompt ─────────────────────────────────────────────
SYSTEM_PROMPT = "You are an executive coach writing for a CIO."

# ── Your question ─────────────────────────────────────────────
question = "Why do enterprise projects delay and how can we reduce delays?"

## Helper Functions

In [33]:
def get_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise ValueError("Missing GROQ_API_KEY in environment or .env file.")
    return Groq(api_key=api_key)


def build_messages(system_prompt: str, question: str, json_mode: bool):
    if json_mode:
        system_prompt = (
            system_prompt.strip()
            + "\n\nReturn only valid JSON. No markdown. No explanation outside JSON."
        )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question},
    ]


def call_model(client, model, messages, temperature, max_tokens, json_mode):
    params = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if json_mode:
        params["response_format"] = {"type": "json_object"}
    return client.chat.completions.create(**params)


def save_log(payload: dict):
    logs_dir = Path("logs")
    logs_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_file = logs_dir / f"run_{timestamp}.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    return out_file

## Run — Ask the Model

In [34]:
def format_answer(text: str, width: int = 80) -> str:
    """Wrap each paragraph/line independently to preserve structure."""
    lines = text.splitlines()
    formatted = []
    for line in lines:
        if line.strip() == "":
            formatted.append("")  # preserve blank lines between paragraphs
        else:
            formatted.extend(textwrap.wrap(line, width=width) or [""])
    return "\n".join(formatted)


client   = get_client()
messages = build_messages(SYSTEM_PROMPT, question, JSON_MODE)

try:
    completion = call_model(
        client=client,
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        json_mode=JSON_MODE,
    )

    answer = completion.choices[0].message.content or ""

    print("=" * 60)
    print(f"  MODEL : {MODEL}")
    print(f"  TEMP  : {TEMPERATURE}   MAX TOKENS : {MAX_TOKENS}")
    print("=" * 60)
    print(f"  Q: {textwrap.fill(question, width=56, subsequent_indent='     ')}")
    print("-" * 60)
    print(format_answer(answer))
    print("=" * 60)

    if SAVE_LOG:
        payload = {
            "timestamp":   datetime.now().isoformat(),
            "model":       MODEL,
            "temperature": TEMPERATURE,
            "max_tokens":  MAX_TOKENS,
            "json_mode":   JSON_MODE,
            "messages":    messages,
            "answer":      answer,
        }
        log_file = save_log(payload)
        print(f"\nSaved log to: {log_file}")

except Exception as e:
    print("\nRequest failed.")
    print(f"Error: {type(e).__name__}: {e}")

  MODEL : llama-3.3-70b-versatile
  TEMP  : 0.8   MAX TOKENS : 300
  Q: Why do enterprise projects delay and how can we reduce
     delays?
------------------------------------------------------------
{
  "delays": [
       {
           "reason": "Poor Project Planning",
           "description": "Inadequate scope definition, unrealistic timelines,
and insufficient resource allocation"
       },
       {
           "reason": "Ineffective Communication",
           "description": "Lack of clear stakeholder expectations, inadequate
status updates, and insufficient issue escalation"
       },
       {
           "reason": "Insufficient Risk Management",
           "description": "Inadequate identification, assessment, and mitigation
of project risks"
       },
       {
           "reason": "Inadequate Resource Allocation",
           "description": "Insufficient skills, inadequate training, and
unrealistic workload expectations"
       }
   ],
   "reduction_strategies": [
       {
       